<a href="https://colab.research.google.com/github/swarankith/ResearchAgent/blob/main/agent_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [4]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [5]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [10]:
# @title User input needed.
# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {

    "cricket": (
        "Cricket is widely regarded as a virtual religion in India, binding millions of passionate fans together. "
        "The historical journey of the sport began in the 18th century when it was first introduced by European merchant sailors. "
        "India achieved its initial major milestone by making its official Test cricket debut at Lord's in 1932. "
        "Over the decades, the nation has evolved from these humble beginnings into an undisputed global powerhouse."
    ),
    "worldcup": (
        "The legacy of Indian cricket is defined by historic World Cup triumphs that transformed the sport's domestic identity. "
        "Under the legendary captaincy of Kapil Dev, India shocked the world by clinching its first-ever World Cup in 1983. "
        "Decades later, Mahendra Singh Dhoni led a golden generation to win the inaugural 2007 T20 World Cup and the 2011 ODI World Cup. "
        "These monumental victories permanently shifted the epicenter of global cricket governance and fandom to the subcontinent."
    ),
    "icc": (
        "The contemporary Indian team operates under a distinct transition in leadership and technical guidance. "
        "Former legendary opener Gautam Gambhir serves as the head coach of the men's national squad across formats. "
        "Young batting sensation Shubman Gill has taken over major leadership responsibilities as the captain for the Test and ODI formats. "
        "This new leadership group balances structural changes while drawing crucial guidance from modern icons like Virat Kohli."
    ),
    "ipl": (
        "The domestic framework of the country is anchored by the massively popular Indian Premier League. "
        "Beyond club franchises, the international bilateral calendar remains incredibly busy for both the men's and women's national teams. "
        "The Board of Control for Cricket in India frequently schedules home and away series to test the bench strength against visiting nations. "
        "Massive global fixtures, such as high-stakes encounters at the Asian Games, continue to showcase the immense depth of talent in the country."
    )
}


In [11]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic - sports, manga, food, music etc. Have a minimum of 4 sentences with each document. Duplicate this for a minimum of 4 documents to test your agent.

[Source: worldcup]
The legacy of Indian cricket is defined by historic World Cup triumphs that transformed the sport's domestic identity. Under the legendary captaincy of Kapil Dev, India shocked the world by clinching its first-ever World Cup in 1983. Decades later, Mahendra Singh Dhoni led a golden generation to win the inaugural 2007 T20 World Cup and the 2011 ODI World Cup. These monumental victories permanently shifted the epicenter of global cricket governance and fandom to the subcontinent.


In [12]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="claude",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/claude


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [17]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

# Also ensure you have a search function defined for your tool to execute
def search_documents(query: str) -> str:
    # This is a basic mock implementation matching your DOCUMENTS dictionary setup
    query_lower = query.lower()
    matches = []
    for doc_id, text in DOCUMENTS.items():
        if doc_id == "sample_protocol":
            continue
        # Quick check if keywords match
        if any(word in text.lower() for word in query_lower.split()):
            matches.append(f"[{doc_id}]: {text}")
    return "\n\n".join(matches) if matches else "No relevant documents found."

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model='gemini-3.5-flash-lite', # Fixed: Changed from model.model_name to string literal
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"

_ = run_agent("after long periopd of time who won the worldcup as captain for india?")


Question: after long periopd of time who won the worldcup as captain for india?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'who won the world cup as captain for india after a long period'})
  Result preview: [cricket]: Cricket is widely regarded as a virtual religion in India, binding millions of passionate...

Final answer:
Based on the history of Indian cricket, India has won the World Cup under two main captains:

1. **Kapil Dev** led India to their historic first-ever World Cup victory in **1983**.
2. After a long period of 28 years, **Mahendra Singh Dhoni** captained India to win the **2011 ICC ODI World Cup** (as well as the inaugural 2007 T20 World Cup). 

If you are referring to the long gap between World Cup titles, **MS Dhoni** is the captain who brought the ODI World Cup back to India after a long period of time (since 1983).


In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)